[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/trim-filter-messages.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239435-lesson-4-trim-and-filter-messages)

# 过滤和裁剪消息

## 回顾

现在，我们对以下几方面有了更深入的理解：

* 如何自定义图状态 Schema
* 如何定义自定义状态 reducer
* 如何使用多个图状态 Schema

## 目标

现在，我们可以开始在 LangGraph 中结合模型使用这些概念了！

在接下来的几节中，我们将构建一个具有长期记忆的聊天机器人。

因为我们的聊天机器人将使用消息，让我们先更多地讨论一下在图状态中处理消息的高级方式。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph langchain_deepseek

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

我们将使用 [LangSmith](https://docs.langchain.com/langsmith/home) 进行[追踪](https://docs.langchain.com/langsmith/observability-concepts)。

我们将日志记录到项目 `langchain-academy`。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

## 消息作为状态

首先，让我们定义一些消息。

In [ ]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage
messages = [AIMessage(f"So you said you were researching ocean mammals?", name="Bot")]
messages.append(HumanMessage(f"Yes, I know about whales. But what others should I learn about?", name="Lance"))

for m in messages:
    m.pretty_print()

回想一下，我们可以将它们传递给聊天模型。

In [ ]:
from langchain_deepseek import ChatDeepSeek
llm = ChatDeepSeek(model="deepseek-chat")
llm.invoke(messages)

我们可以在一个使用 `MessagesState` 的简单图中运行我们的聊天模型。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph, START, END

# 节点
def chat_model_node(state: MessagesState):
    return {"messages": llm.invoke(state["messages"])}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("chat_model", chat_model_node)
builder.add_edge(START, "chat_model")
builder.add_edge("chat_model", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
output = graph.invoke({'messages': messages})
for m in output['messages']:
    m.pretty_print()

## Reducer

处理消息时的一个实际挑战是管理长时间运行的对话。

如果我们不小心，长时间运行的对话会导致高 token 使用和延迟，因为我们向模型传递了不断增长的消息列表。

我们有几种方法来解决这个问题。

首先，回想一下我们使用 `RemoveMessage` 和 `add_messages` reducer 的技巧。

In [ ]:
from langchain_core.messages import RemoveMessage

# 节点
def filter_messages(state: MessagesState):
    # 删除除最近 2 条消息之外的所有消息
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"messages": delete_messages}

def chat_model_node(state: MessagesState):    
    return {"messages": [llm.invoke(state["messages"])]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("filter", filter_messages)
builder.add_node("chat_model", chat_model_node)
builder.add_edge(START, "filter")
builder.add_edge("filter", "chat_model")
builder.add_edge("chat_model", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# 带有前置消息的消息列表
messages = [AIMessage("Hi.", name="Bot", id="1")]
messages.append(HumanMessage("Hi.", name="Lance", id="2"))
messages.append(AIMessage("So you said you were researching ocean mammals?", name="Bot", id="3"))
messages.append(HumanMessage("Yes, I know about whales. But what others should I learn about?", name="Lance", id="4"))

# 调用
output = graph.invoke({'messages': messages})
for m in output['messages']:
    m.pretty_print()

## 过滤消息

如果你不需要或不想修改图状态，你可以只过滤传递给聊天模型的消息。

例如，只需将过滤后的列表传递给模型：`llm.invoke(messages[-1:])`。

In [ ]:
# 节点
def chat_model_node(state: MessagesState):
    return {"messages": [llm.invoke(state["messages"][-1:])]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("chat_model", chat_model_node)
builder.add_edge(START, "chat_model")
builder.add_edge("chat_model", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

让我们使用现有的消息列表，追加上面的 LLM 回复，然后再追加一个跟进问题。

In [ ]:
messages.append(output['messages'][-1])
messages.append(HumanMessage(f"Tell me more about Narwhals!", name="Lance"))

In [ ]:
for m in messages:
    m.pretty_print()

In [ ]:
# 调用，使用消息过滤
output = graph.invoke({'messages': messages})
for m in output['messages']:
    m.pretty_print()

状态中包含了所有消息。

但是，让我们查看 LangSmith 追踪记录，可以看到模型调用只使用了最后一条消息：

https://smith.langchain.com/public/75aca3ce-ef19-4b92-94be-0178c7a660d9/r

## 裁剪消息

另一种方法是根据指定的 token 数量来[裁剪消息](https://docs.langchain.com/oss/python/langgraph/add-memory#trim-messages)。

这将消息历史限制在指定的 token 数量内。

过滤只是返回 Agent 之间消息的事后子集，而裁剪则限制了聊天模型可以用来响应的 token 数量。

请参阅下面的 `trim_messages`。

In [ ]:
from langchain_core.messages import trim_messages

# 节点
def chat_model_node(state: MessagesState):
    messages = trim_messages(
            state["messages"],
            max_tokens=100,
            strategy="last",
            token_counter=ChatDeepSeek(model="deepseek-chat"),
            allow_partial=False,
        )
    return {"messages": [llm.invoke(messages)]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("chat_model", chat_model_node)
builder.add_edge(START, "chat_model")
builder.add_edge("chat_model", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
messages.append(output['messages'][-1])
messages.append(HumanMessage(f"Tell me where Orcas live!", name="Lance"))

In [ ]:
# 裁剪消息的示例
trim_messages(
            messages,
            max_tokens=100,
            strategy="last",
            token_counter=ChatDeepSeek(model="deepseek-chat"),
            allow_partial=False
        )

In [ ]:
# 调用，在 chat_model_node 中使用消息裁剪
messages_out_trim = graph.invoke({'messages': messages})

让我们查看 LangSmith 追踪记录，以了解模型调用：

https://smith.langchain.com/public/b153f7e9-f1a5-4d60-8074-f0d7ab5b42ef/r